[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/e8/e8.ipynb)

# E8 Reflections

This example computes the 240 roots of the exceptional Lie algebra E8 in eight dimensions and projects their orbits onto its four invariant rotation planes. Each root is modelled as a mirror hyperplane through the origin whose reflection permutes the entire root system. Eight generating reflections multiply into a single rotation that closes after thirty steps while turning four mutually perpendicular planes simultaneously.

Multiplying the eight unit mirror vectors forms the Coxeter rotor `reduce(mul, mirrors)`, whose sandwich action `coxeter >> Vector` captures the eight-dimensional rotation as a single linear map. Averaging that map with its transpose via `(action + action.adjoint()) / 2` cancels the skew turn in each invariant plane and leaves the cosine of its rotation angle, so one real symmetric eigensolve recovers all four orthogonal plane bivectors `plane_x ^ plane_y` and their commuting generator `-(planes * rates).sum(axis=0)`. Pierre-Philippe Dechant develops this rotor construction in [The E8 Geometry from a Clifford Perspective](https://eprints.whiterose.ac.uk/id/eprint/96267/1/E8Geometry.pdf).

In matrix Coxeter-group notation, each simple root $\alpha_k \in \mathbb{R}^8$ defines a Householder matrix $R_k = I_8 - 2\,\alpha_k \alpha_k^\top / (\alpha_k^\top \alpha_k)$, and the Coxeter matrix $W = R_1 \cdots R_8$ is diagonalized over $\mathbb{C}^8$ as $W v_m = e^{2\pi i m / h} v_m$ for $h = 30$ and exponents $m \in \{1, 7, 11, 13, 17, 19, 23, 29\}$. Real invariant planes and continuous interpolations are then reassembled from conjugate eigenvector pairs via $\operatorname{Re} v_m$, $\operatorname{Im} v_m$, and block-diagonal $2\times 2$ rotation matrices $\exp(\theta_a J_a)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce
from operator import mul

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext, stack
from examples.math.e8 import render
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=6, suppress=True)

ga = Algebra("a+b+c+d+e+f+g+h+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Vector = ga.gatype.vector()                                                   # Vector+
Bivector = ga.gatype.bivector()                                               # Bivector+
Rotor = ga.gatype.rotor()                                                     # Rotor+
VectorMap = ga.gatype((Vector, Vector))                                       # Vector+ <- Vector+
coxeter_order = 30

## Root orbits

Eight generating mirrors, which are called simple roots, generate the full reflection group of E8. Their geometric product forms the Coxeter rotor, an eight-dimensional rotation that returns to the identity after thirty steps. Starting from one suitably chosen seed root per mirror, thirty applications of the Coxeter rotor partition all 240 roots into eight disjoint orbits.

In [ ]:
n_steps = coxeter_order
simple_roots = stack([
    (mv.a + mv.h - mv.b - mv.c - mv.d - mv.e - mv.f - mv.g) / 2,
    mv.a + mv.b,
    mv.b - mv.a,
    mv.c - mv.b,
    mv.d - mv.c,
    mv.e - mv.d,
    mv.f - mv.e,
    mv.g - mv.f,
])                                                                            # [8] Vector+

In [ ]:
def reflection_seeds(roots: Vector) -> Iterator[Vector]:
    """Yields one seed root (`[] Vector+`) per Coxeter orbit by reflecting each mirror in the mirrors before it."""
    prefix = Vector                                                           # [] Vector+ <- Vector+
    for root in roots:
        yield prefix(root)                                                    # [] Vector+
        # Composing on the right applies earlier mirrors after later ones, matching the Coxeter product order.
        reflection = root.normalized() >> Vector                              # [] Vector+ <- Vector+
        prefix = prefix(reflection)                                           # [] Vector+ <- Vector+


def orbit(action: VectorMap, seeds: Vector, n_steps: int) -> Iterator[Vector]:
    """Yields `n_steps` successive images (`[...] Vector+`) of `seeds` under `action`."""
    for _ in range(n_steps):
        yield seeds                                                           # [...] Vector+
        seeds = action(seeds)                                                 # [...] Vector+


mirrors = simple_roots.normalized()                                           # [8] Vector+
coxeter = reduce(mul, mirrors)                                                # [] Rotor+
action = coxeter >> Vector                                                    # [] Vector+ <- Vector+
seeds = stack(reflection_seeds(simple_roots))                                 # [8] Vector+
roots = stack(orbit(action, seeds, n_steps))                                  # [30, 8] Vector+

## The Coxeter plane

Forward and backward Coxeter steps turn each invariant plane by equal and opposite angles, so their sideways components cancel when averaged. Each invariant plane is therefore an eigenspace of the symmetrized map scaled by the cosine of its step angle, and the slowest of the four orthogonal planes is called the Coxeter plane.

In [ ]:
def invariant_planes(action: VectorMap, n_steps: int) -> tuple[Vector, Vector, Bivector, Scalar]:
    """Symmetric eigendecomposition of `action` isolates the four invariant plane frames, bivectors, and turn counts (`[4]`)."""
    symmetric = (action + action.adjoint()) / 2                               # [] Vector+ <- Vector+
    cosines, eigenvectors = symmetric.eigh()                                  # [8] Scalar+, Vector+
    # Equal eigenvalues occur in pairs; descending cosines order the four planes from slow to fast.
    plane_x = eigenvectors[::-2]                                              # [4] Vector+
    # Subtracting the projection along plane_x isolates the perpendicular turn direction in each plane.
    plane_y = (action(plane_x) - cosines[::-2] * plane_x).normalized()        # [4] Vector+
    planes = plane_x ^ plane_y                                                # [4] Bivector+
    rates = cosines[::-2].arccos() * n_steps / (2 * np.pi)                    # [4] Scalar+
    return plane_x, plane_y, planes, rates


plane_x, plane_y, planes, rates = invariant_planes(action, n_steps)           # [4] Vector+, Vector+, Bivector+, Scalar+
print("Turns in 30 steps:", rates.to_array())
display(Image(filename=str(save_figure(
    render.draw_roots(roots, plane_x[0], plane_y[0]), "e8_roots",
))));

In the Coxeter plane, the eight thirty-root orbits project onto eight concentric rings coloured by orbit, while faint segments join pairs of roots at minimum eight-dimensional separation.

## Four simultaneous turns

Because the four invariant planes are mutually orthogonal, their unit bivectors commute and generate independent planar rotations simultaneously. Summing those four bivectors weighted by their winding rates yields the logarithm of the Coxeter rotor, whose fractional powers interpolate a smooth eight-dimensional rotation through the discrete root symmetries. Rendering the 270 animation frames takes about fifteen seconds.

In [ ]:
n_substeps = 9
n_frames = n_steps * n_substeps
frame_duration_ms = 40

In [ ]:
def step_rotor(planes: Bivector, rates: Scalar, n_steps: int, n_substeps: int) -> Rotor:
    """Commuting plane generators (`[4] Bivector+`) exponentiate to the fractional-step rotor (`[] Rotor+`)."""
    generator = -(planes * rates).sum(axis=0) * np.pi / n_steps               # [] Bivector+
    return (generator / n_substeps).exp()                                     # [] Rotor+


rotor = step_rotor(planes, rates, n_steps, n_substeps)                        # [] Rotor+
advance = rotor >> Vector                                                     # [] Vector+ <- Vector+
orbit_history = show_states(
    orbit(advance, seeds, n_frames),
    render.planes_animation,
    roots,
    plane_x,
    plane_y,
    n_substeps,
    name="e8_planes",
    duration_ms=frame_duration_ms,
);

Across the four panels from left to right, each coloured rider follows one seed root as it completes 1, 7, 11, and 13 revolutions over a full thirty-step cycle. Faint dots mark the discrete roots on each orbit circle, and equal-duration trails span wider arcs in the faster planes.

In [ ]:
# checks
flat_roots = roots.reshape((-1,))                                             # [240] Vector+
plane_gram = planes[:, None] | planes[None, :]                                # [4, 4] Scalar+

assert len(flat_roots) == 240
assert ((flat_roots.scalar_norm_squared() - 2).abs() < 1e-10).all()
assert ((flat_roots[:, None] - flat_roots[None, :]).scalar_norm_squared() < 1e-6).sum() == 240
assert (((coxeter >> roots[-1]) - seeds).scalar_norm_squared() < 1e-10).all()
assert ((plane_gram + np.eye(4)).abs() < 1e-10).all()
assert ((rates - np.array([1, 7, 11, 13])).abs() < 1e-10).all()
assert ((stack(orbit_history[::n_substeps]) - roots).scalar_norm_squared() < 1e-10).all()